# Shortcut Learning in Risky-Intent Classification


## Experiment List

| ID | Experiment |
|---|---|
| E1 | RoBERTa Fine-Tuned Baseline |
| E2 | RoBERTa + Keyword Masking |
| E3 | RoBERTa + Counterfactual Augmentation |
| E4 | RoBERTa + Keyword Masking + Counterfactual Augmentation |
| E5 | RoBERTa + Experience Replay |
| E6 | NLI Zero-Shot |
| E7 | NLI Fine-Tuned |
| E8 | NLI + Keyword Masking + Counterfactual Augmentation |
| E9 | NLI + Experience Replay + Keyword Masking + Counterfactual Augmentation |

Each experiment shows:

- Metrics
- Classification report
- Confusion matrix
- OOD category performance
- Wrong examples
- Confident wrong examples
- SHAP analysis on three fixed sentences when applicable

Fixed SHAP sentences:

```python
sentences = [
    "I eat ice cream.",
    "I cut myself.",
    "I want to die to make everyone happy in this house.",
]

# 1. Clone Repository and Setup

This section is for running the notebook on a fresh Colab or GPU instance.

If the notebook is already inside the repository, skip the `git clone` command and only run the installation.

In [ ]:
!git clone https://github.com/juliairsalina/shortcut-learning-risky-intent.git
%cd shortcut-learning-risky-intent
!pip install -r requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training may be slow.")

# 2. Imports, Paths, and Global Settings

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display, Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 80)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
ROOT = Path(".")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RESULTS_DIR = ROOT / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
PLOTS_DIR = RESULTS_DIR / "plots"
SHAP_DIR = RESULTS_DIR / "shap"

SAVED_MODELS_DIR = ROOT / "saved_models"

for folder in [
    PROCESSED_DIR,
    METRICS_DIR,
    PRED_DIR,
    PLOTS_DIR,
    SHAP_DIR,
    SAVED_MODELS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train.csv"
VAL_PATH = PROCESSED_DIR / "val.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
OOD_PATH = PROCESSED_DIR / "ood.csv"

TRAIN_MASKED_PATH = PROCESSED_DIR / "train_masked.csv"
TRAIN_COUNTERFACTUAL_PATH = PROCESSED_DIR / "train_counterfactual.csv"
TRAIN_FULL_PATH = PROCESSED_DIR / "train_full.csv"
REPLAY_PATH = PROCESSED_DIR / "replay_examples.csv"

ROBERTA_MODEL = "roberta-base"
NLI_MODEL = "cross-encoder/nli-roberta-base"

MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

LABEL_MAP = {
    0: "non-risky",
    1: "risky",
}

LABEL_TO_ID = {
    "non-risky": 0,
    "risky": 1,
    0: 0,
    1: 1,
}

SHAP_SENTENCES = [
    "I eat ice cream.",
    "I cut myself.",
    "I want to die to make everyone happy in this house.",
]

# 3. File Check

In [ ]:
required_files = [
    RAW_DIR / "dataset_nad.xlsx",
    RAW_DIR / "custom_ood_set_150_julia.csv",
    Path("src/data_utils.py"),
    Path("src/masking.py"),
    Path("src/augmentation.py"),
]

print("File check:")

for path in required_files:
    if path.exists():
        print(f"FOUND   : {path}")
    else:
        print(f"MISSING : {path}")

# 4. Helper Functions

In [ ]:
def run_command(command):
    print("=" * 80)
    print(command)
    print("=" * 80)

    result = subprocess.run(command, shell=True)

    if result.returncode != 0:
        print(f"WARNING: Command failed: {command}")

    return result.returncode


def load_csv(path):
    path = Path(path)

    if not path.exists():
        print(f"WARNING: Missing CSV file: {path}")
        return None

    return pd.read_csv(path)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def load_json(path):
    path = Path(path)

    if not path.exists():
        print(f"WARNING: Missing JSON file: {path}")
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_label_column(df):
    df = df.copy()

    if "gold_label" in df.columns and "label" not in df.columns:
        df["label"] = df["gold_label"]

    if "label" not in df.columns:
        raise ValueError("Dataset must contain label or gold_label column.")

    df["label"] = df["label"].map(
        lambda x: LABEL_TO_ID[x] if x in LABEL_TO_ID else int(x)
    )

    df["label"] = df["label"].astype(int)

    return df


def show_dataset_overview(name, df):
    if df is None:
        return

    print("=" * 80)
    print(name)
    print("=" * 80)

    print("Shape:", df.shape)
    display(df.head())

    if "label" in df.columns:
        label_counts = df["label"].map(LABEL_MAP).value_counts().reset_index()
        label_counts.columns = ["label", "count"]
        display(label_counts)

    if "category" in df.columns:
        cat_counts = df["category"].value_counts().reset_index()
        cat_counts.columns = ["category", "count"]
        display(cat_counts)


def print_experiment_header(title):
    print("\n")
    print("=" * 80)
    print(f"  {title}")
    print("=" * 80)


def print_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["non-risky", "risky"],
            zero_division=0,
        )
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=["non-risky", "risky"],
            output_dict=True,
            zero_division=0,
        ),
    }


def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.xticks([0, 1], ["non-risky", "risky"])
    plt.yticks([0, 1], ["non-risky", "risky"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()

    return cm.tolist()


def save_prediction_csv(pred_df, experiment_id, split_name):
    path = PRED_DIR / f"{experiment_id}_{split_name}_predictions.csv"
    pred_df.to_csv(path, index=False)

    print(f"Saved predictions to: {path}")

    return path


def show_wrong_examples(pred_df, n=10):
    wrong = pred_df[pred_df["true_label"] != pred_df["predicted_label"]].copy()

    print(f"Wrong examples: {len(wrong)}")

    if "confidence" in wrong.columns:
        wrong = wrong.sort_values("confidence", ascending=False)

    display(wrong.head(n))

    return wrong


def show_confident_wrong_examples(pred_df, n=10, threshold=0.8):
    wrong = pred_df[pred_df["true_label"] != pred_df["predicted_label"]].copy()

    if "confidence" not in wrong.columns:
        print("No confidence column found.")
        display(wrong.head(n))
        return wrong

    confident_wrong = wrong[wrong["confidence"] >= threshold].copy()
    confident_wrong = confident_wrong.sort_values("confidence", ascending=False)

    print(f"Confident wrong examples, confidence >= {threshold}: {len(confident_wrong)}")
    display(confident_wrong.head(n))

    return confident_wrong


def evaluate_category_metrics(pred_df):
    if "category" not in pred_df.columns:
        return None

    rows = []

    for category, group in pred_df.groupby("category"):
        rows.append({
            "category": category,
            "count": len(group),
            "accuracy": accuracy_score(group["true_label"], group["predicted_label"]),
            "macro_f1": f1_score(
                group["true_label"],
                group["predicted_label"],
                average="macro",
                zero_division=0,
            ),
        })

    category_df = pd.DataFrame(rows)

    print("OOD Category Metrics:")
    display(category_df)

    plt.figure(figsize=(8, 4))
    plt.bar(category_df["category"], category_df["macro_f1"])
    plt.title("OOD Macro-F1 by Category")
    plt.xlabel("Category")
    plt.ylabel("Macro-F1")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    return category_df.to_dict(orient="records")

# 5. Data Preparation

This section creates:

- train.csv
- val.csv
- test.csv
- ood.csv
- train_masked.csv
- train_counterfactual.csv
- train_full.csv

In [ ]:
run_command("python -m src.data_utils")
run_command("python -m src.masking")
run_command("python -m src.augmentation")

In [ ]:
train_df = normalize_label_column(load_csv(TRAIN_PATH))
val_df = normalize_label_column(load_csv(VAL_PATH))
test_df = normalize_label_column(load_csv(TEST_PATH))
ood_df = normalize_label_column(load_csv(OOD_PATH))

train_masked_df = normalize_label_column(load_csv(TRAIN_MASKED_PATH))
train_counterfactual_df = normalize_label_column(load_csv(TRAIN_COUNTERFACTUAL_PATH))
train_full_df = normalize_label_column(load_csv(TRAIN_FULL_PATH))

show_dataset_overview("Train", train_df)
show_dataset_overview("Validation", val_df)
show_dataset_overview("Test", test_df)
show_dataset_overview("OOD", ood_df)

show_dataset_overview("Train Masked", train_masked_df)
show_dataset_overview("Train Counterfactual", train_counterfactual_df)
show_dataset_overview("Train Full", train_full_df)

# 6. Create Experience Replay Dataset

Experience replay repeatedly shows the model hard examples involving:

- negation
- temporal recovery
- figurative language
- normal usage of risky keywords
- clear risky intent

In [ ]:
replay_examples = [
    ("I do not want to die.", "die", 0, "negation"),
    ("I don't want to hurt myself.", "hurt", 0, "negation"),
    ("I will not kill myself.", "kill", 0, "negation"),
    ("I won't do anything stupid tonight.", "stupid", 0, "negation"),
    ("I no longer want to die.", "die", 0, "negation_temporal"),
    ("I stopped wanting to hurt myself.", "hurt", 0, "negation_temporal"),
    ("I used to want to disappear, but not anymore.", "disappear", 0, "temporal"),
    ("I finally stopped cutting myself.", "cut", 0, "temporal"),

    ("I want to die of laughter.", "die", 0, "figurative"),
    ("I am dying of embarrassment.", "die", 0, "figurative"),
    ("This homework is killing me.", "kill", 0, "figurative"),
    ("I killed it at the interview.", "kill", 0, "figurative"),
    ("I cut the paper into two pieces.", "cut", 0, "normal"),
    ("I cut my hair yesterday.", "cut", 0, "normal"),
    ("I jumped for joy.", "jump", 0, "figurative"),

    ("I want to die tonight.", "die", 1, "risky"),
    ("I want to hurt myself tonight.", "hurt", 1, "risky"),
    ("I am going to kill myself.", "kill", 1, "risky"),
    ("I want to cut myself.", "cut", 1, "risky"),
    ("I might jump off the building.", "jump", 1, "risky"),
    ("I want to die to make everyone happy in this house.", "die", 1, "risky"),
]

replay_df = pd.DataFrame(
    replay_examples,
    columns=["text", "keyword", "label", "category"],
)

replay_df.to_csv(REPLAY_PATH, index=False)

print(f"Saved replay examples to: {REPLAY_PATH}")
show_dataset_overview("Replay Examples", replay_df)

# 7. Dataset and Trainer Classes

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


def build_training_args(output_dir):
    return TrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=20,
        save_total_limit=2,
        report_to="none",
        seed=SEED,
    )


def compute_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
    }


def build_trainer(model, args, train_dataset, eval_dataset, tokenizer):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_trainer_metrics,
    )

# 8. Training and Evaluation Functions

In [ ]:
def train_finetuned_model(
    experiment_id,
    experiment_name,
    model_name,
    train_data,
    val_data,
    use_replay=False,
    replay_data=None,
    replay_repeat=5,
):
    print_experiment_header(f"Training {experiment_name}")

    set_seed(SEED)

    train_data = train_data.copy()

    if use_replay:
        if replay_data is None:
            raise ValueError("Replay data is required when use_replay=True.")

        repeated_replay = pd.concat(
            [replay_data.copy() for _ in range(replay_repeat)],
            ignore_index=True,
        )

        train_data = pd.concat(
            [train_data, repeated_replay],
            ignore_index=True,
        ).sample(frac=1, random_state=SEED).reset_index(drop=True)

        print("Experience Replay enabled.")
        print("Replay repeat:", replay_repeat)
        print("Training rows after replay:", len(train_data))

    output_dir = SAVED_MODELS_DIR / experiment_id
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True,
    )

    train_dataset = TextDataset(
        texts=train_data["text"],
        labels=train_data["label"],
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    val_dataset = TextDataset(
        texts=val_data["text"],
        labels=val_data["label"],
        tokenizer=tokenizer,
        max_len=MAX_LEN,
    )

    args = build_training_args(output_dir)

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    train_log = {
        "experiment_id": experiment_id,
        "experiment_name": experiment_name,
        "model_name": model_name,
        "train_rows": len(train_data),
        "validation_rows": len(val_data),
        "use_replay": use_replay,
        "replay_repeat": replay_repeat if use_replay else 0,
        "log_history": trainer.state.log_history,
    }

    save_json(train_log, METRICS_DIR / f"{experiment_id}_train_log.json")

    print(f"Saved model to: {output_dir}")

    return output_dir


def predict_with_finetuned_model(experiment_id, data_df, split_name):
    model_path = SAVED_MODELS_DIR / experiment_id

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    texts = list(data_df["text"])
    true_labels = list(data_df["label"])

    all_preds = []
    all_confs = []
    all_risky_probs = []

    with torch.no_grad():
        for text in texts:
            encoding = tokenizer(
                str(text),
                truncation=True,
                padding="max_length",
                max_length=MAX_LEN,
                return_tensors="pt",
            )

            encoding = {k: v.to(DEVICE) for k, v in encoding.items()}

            outputs = model(**encoding)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

            pred = int(np.argmax(probs))
            conf = float(np.max(probs))
            risky_prob = float(probs[1])

            all_preds.append(pred)
            all_confs.append(conf)
            all_risky_probs.append(risky_prob)

    pred_df = data_df.copy()
    pred_df["true_label"] = true_labels
    pred_df["true_label_name"] = pred_df["true_label"].map(LABEL_MAP)
    pred_df["predicted_label"] = all_preds
    pred_df["prediction"] = pred_df["predicted_label"].map(LABEL_MAP)
    pred_df["confidence"] = all_confs
    pred_df["risky_probability"] = all_risky_probs
    pred_df["correct"] = pred_df["true_label"] == pred_df["predicted_label"]

    save_prediction_csv(pred_df, experiment_id, split_name)

    return pred_df


def evaluate_finetuned_experiment(experiment_id, experiment_name):
    print_experiment_header(experiment_name)

    id_pred_df = predict_with_finetuned_model(experiment_id, test_df, "id")
    ood_pred_df = predict_with_finetuned_model(experiment_id, ood_df, "ood")

    print("\nID Test Results")
    id_metrics = print_metrics(
        id_pred_df["true_label"],
        id_pred_df["predicted_label"],
    )

    id_cm = plot_confusion_matrix(
        id_pred_df["true_label"],
        id_pred_df["predicted_label"],
        title=f"{experiment_id} ID Confusion Matrix",
        save_path=PLOTS_DIR / f"{experiment_id}_id_confusion_matrix.png",
    )

    print("\nOOD Results")
    ood_metrics = print_metrics(
        ood_pred_df["true_label"],
        ood_pred_df["predicted_label"],
    )

    ood_cm = plot_confusion_matrix(
        ood_pred_df["true_label"],
        ood_pred_df["predicted_label"],
        title=f"{experiment_id} OOD Confusion Matrix",
        save_path=PLOTS_DIR / f"{experiment_id}_ood_confusion_matrix.png",
    )

    print("\nOOD Category Metrics")
    category_metrics = evaluate_category_metrics(ood_pred_df)

    print("\nWrong OOD Examples")
    show_wrong_examples(ood_pred_df, n=10)

    print("\nConfident Wrong OOD Examples")
    show_confident_wrong_examples(ood_pred_df, n=10)

    metrics = {
        "experiment_id": experiment_id,
        "experiment_name": experiment_name,
        "id_test": {
            **id_metrics,
            "confusion_matrix": id_cm,
            "confident_wrong_count": int(
                len(id_pred_df[
                    (id_pred_df["correct"] == False) &
                    (id_pred_df["confidence"] >= 0.8)
                ])
            ),
        },
        "ood": {
            **ood_metrics,
            "confusion_matrix": ood_cm,
            "confident_wrong_count": int(
                len(ood_pred_df[
                    (ood_pred_df["correct"] == False) &
                    (ood_pred_df["confidence"] >= 0.8)
                ])
            ),
            "category_metrics": category_metrics,
        },
    }

    save_json(metrics, METRICS_DIR / f"{experiment_id}.json")

    print(f"Saved metrics to: {METRICS_DIR / f'{experiment_id}.json'}")

    return metrics, id_pred_df, ood_pred_df

# 9. SHAP Helper Function

SHAP is used on three fixed sentences:

1. `I eat ice cream.`
2. `I cut myself.`
3. `I want to die to make everyone happy in this house.`

In [ ]:
def run_shap_for_finetuned_model(experiment_id, sentences=SHAP_SENTENCES):
    print_experiment_header(f"SHAP Analysis: {experiment_id}")

    try:
        import shap
    except ImportError:
        print("SHAP is not installed. Installing SHAP...")
        run_command("pip install shap")
        import shap

    model_path = SAVED_MODELS_DIR / experiment_id

    if not model_path.exists():
        print(f"Model path does not exist: {model_path}")
        return

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    clf = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=0 if DEVICE == "cuda" else -1,
        top_k=None,
    )

    try:
        explainer = shap.Explainer(clf)

        for i, sentence in enumerate(sentences):
            print("=" * 80)
            print(f"Sentence {i + 1}: {sentence}")
            print("=" * 80)

            shap_values = explainer([sentence])
            shap.plots.text(shap_values[0])

    except Exception as e:
        print("SHAP failed for this model.")
        print("Reason:", e)
        print("You can still use metrics and confusion matrix for this experiment.")

# 10. NLI Zero-Shot Helper Function

In [ ]:
def nli_zero_shot_predict(texts):
    zero_shot = pipeline(
        "zero-shot-classification",
        model=NLI_MODEL,
        device=0 if DEVICE == "cuda" else -1,
    )

    candidate_labels = [
        "non-risky intent",
        "risky intent",
    ]

    predictions = []
    confidences = []
    risky_probs = []

    for text in texts:
        result = zero_shot(
            str(text),
            candidate_labels=candidate_labels,
            hypothesis_template="This text expresses {}.",
        )

        label = result["labels"][0]
        confidence = float(result["scores"][0])

        score_map = dict(zip(result["labels"], result["scores"]))
        risky_score = float(score_map.get("risky intent", 0.0))

        pred = 1 if label == "risky intent" else 0

        predictions.append(pred)
        confidences.append(confidence)
        risky_probs.append(risky_score)

    return predictions, confidences, risky_probs


def evaluate_zero_shot_nli(experiment_id="E6", experiment_name="NLI Zero-Shot"):
    print_experiment_header(experiment_name)

    id_preds, id_confs, id_risky_probs = nli_zero_shot_predict(test_df["text"])
    ood_preds, ood_confs, ood_risky_probs = nli_zero_shot_predict(ood_df["text"])

    id_pred_df = test_df.copy()
    id_pred_df["true_label"] = id_pred_df["label"]
    id_pred_df["true_label_name"] = id_pred_df["true_label"].map(LABEL_MAP)
    id_pred_df["predicted_label"] = id_preds
    id_pred_df["prediction"] = id_pred_df["predicted_label"].map(LABEL_MAP)
    id_pred_df["confidence"] = id_confs
    id_pred_df["risky_probability"] = id_risky_probs
    id_pred_df["correct"] = id_pred_df["true_label"] == id_pred_df["predicted_label"]

    ood_pred_df = ood_df.copy()
    ood_pred_df["true_label"] = ood_pred_df["label"]
    ood_pred_df["true_label_name"] = ood_pred_df["true_label"].map(LABEL_MAP)
    ood_pred_df["predicted_label"] = ood_preds
    ood_pred_df["prediction"] = ood_pred_df["predicted_label"].map(LABEL_MAP)
    ood_pred_df["confidence"] = ood_confs
    ood_pred_df["risky_probability"] = ood_risky_probs
    ood_pred_df["correct"] = ood_pred_df["true_label"] == ood_pred_df["predicted_label"]

    save_prediction_csv(id_pred_df, experiment_id, "id")
    save_prediction_csv(ood_pred_df, experiment_id, "ood")

    print("\nID Test Results")
    id_metrics = print_metrics(id_pred_df["true_label"], id_pred_df["predicted_label"])

    id_cm = plot_confusion_matrix(
        id_pred_df["true_label"],
        id_pred_df["predicted_label"],
        title=f"{experiment_id} ID Confusion Matrix",
        save_path=PLOTS_DIR / f"{experiment_id}_id_confusion_matrix.png",
    )

    print("\nOOD Results")
    ood_metrics = print_metrics(ood_pred_df["true_label"], ood_pred_df["predicted_label"])

    ood_cm = plot_confusion_matrix(
        ood_pred_df["true_label"],
        ood_pred_df["predicted_label"],
        title=f"{experiment_id} OOD Confusion Matrix",
        save_path=PLOTS_DIR / f"{experiment_id}_ood_confusion_matrix.png",
    )

    print("\nOOD Category Metrics")
    category_metrics = evaluate_category_metrics(ood_pred_df)

    print("\nWrong OOD Examples")
    show_wrong_examples(ood_pred_df, n=10)

    print("\nConfident Wrong OOD Examples")
    show_confident_wrong_examples(ood_pred_df, n=10)

    metrics = {
        "experiment_id": experiment_id,
        "experiment_name": experiment_name,
        "id_test": {
            **id_metrics,
            "confusion_matrix": id_cm,
            "confident_wrong_count": int(
                len(id_pred_df[
                    (id_pred_df["correct"] == False) &
                    (id_pred_df["confidence"] >= 0.8)
                ])
            ),
        },
        "ood": {
            **ood_metrics,
            "confusion_matrix": ood_cm,
            "confident_wrong_count": int(
                len(ood_pred_df[
                    (ood_pred_df["correct"] == False) &
                    (ood_pred_df["confidence"] >= 0.8)
                ])
            ),
            "category_metrics": category_metrics,
        },
    }

    save_json(metrics, METRICS_DIR / f"{experiment_id}.json")

    print("Note: SHAP is skipped for zero-shot NLI because SHAP support for zero-shot-classification pipeline is unstable.")

    return metrics, id_pred_df, ood_pred_df

# 11. Full Experiment Runner Function

In [ ]:
def run_finetuned_experiment(
    experiment_id,
    experiment_name,
    model_name,
    train_data,
    use_replay=False,
    replay_data=None,
    replay_repeat=5,
    run_training=True,
    run_shap=True,
):
    if run_training:
        train_finetuned_model(
            experiment_id=experiment_id,
            experiment_name=experiment_name,
            model_name=model_name,
            train_data=train_data,
            val_data=val_df,
            use_replay=use_replay,
            replay_data=replay_data,
            replay_repeat=replay_repeat,
        )
    else:
        print(f"Skipping training for {experiment_id}. Using existing saved model.")

    metrics, id_pred_df, ood_pred_df = evaluate_finetuned_experiment(
        experiment_id=experiment_id,
        experiment_name=experiment_name,
    )

    if run_shap:
        run_shap_for_finetuned_model(
            experiment_id=experiment_id,
            sentences=SHAP_SENTENCES,
        )

    return metrics, id_pred_df, ood_pred_df

# 12. E1: RoBERTa Fine-Tuned Baseline

This is the main RoBERTa baseline.

Training data:

`data/processed/train.csv`

In [ ]:
e1_metrics, e1_id_preds, e1_ood_preds = run_finetuned_experiment(
    experiment_id="E1",
    experiment_name="RoBERTa Fine-Tuned Baseline",
    model_name=ROBERTA_MODEL,
    train_data=train_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 13. E2: RoBERTa + Keyword Masking

This experiment tests whether masking shortcut-trigger words reduces keyword reliance.

Training data:

`data/processed/train_masked.csv`

In [ ]:
e2_metrics, e2_id_preds, e2_ood_preds = run_finetuned_experiment(
    experiment_id="E2",
    experiment_name="RoBERTa + Keyword Masking",
    model_name=ROBERTA_MODEL,
    train_data=train_masked_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 14. E3: RoBERTa + Counterfactual Augmentation

This experiment tests whether counterfactual augmentation improves context understanding.

Training data:

`data/processed/train_counterfactual.csv`

In [ ]:
e3_metrics, e3_id_preds, e3_ood_preds = run_finetuned_experiment(
    experiment_id="E3",
    experiment_name="RoBERTa + Counterfactual Augmentation",
    model_name=ROBERTA_MODEL,
    train_data=train_counterfactual_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 15. E4: RoBERTa + Keyword Masking + Counterfactual Augmentation

This experiment combines the two data-level mitigation methods.

Training data:

`data/processed/train_full.csv`

In [ ]:
e4_metrics, e4_id_preds, e4_ood_preds = run_finetuned_experiment(
    experiment_id="E4",
    experiment_name="RoBERTa + Keyword Masking + Counterfactual Augmentation",
    model_name=ROBERTA_MODEL,
    train_data=train_full_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 16. E5: RoBERTa + Experience Replay

This experiment tests whether replaying hard examples improves robustness.

Training data:

`data/processed/train_full.csv`

Replay data:

`data/processed/replay_examples.csv`

In [ ]:
e5_metrics, e5_id_preds, e5_ood_preds = run_finetuned_experiment(
    experiment_id="E5",
    experiment_name="RoBERTa + Experience Replay",
    model_name=ROBERTA_MODEL,
    train_data=train_full_df,
    use_replay=True,
    replay_data=replay_df,
    replay_repeat=5,
    run_training=True,
    run_shap=True,
)

# 17. E6: NLI Zero-Shot

This experiment uses an NLI model directly with zero-shot classification.

Model:

`cross-encoder/nli-roberta-base`

No fine-tuning is performed.

In [ ]:
e6_metrics, e6_id_preds, e6_ood_preds = evaluate_zero_shot_nli(
    experiment_id="E6",
    experiment_name="NLI Zero-Shot",
)

# 18. E7: NLI Fine-Tuned

This experiment fine-tunes an NLI-trained RoBERTa model on the risky-intent dataset.

Model:

`cross-encoder/nli-roberta-base`

Training data:

`data/processed/train.csv`

In [ ]:
e7_metrics, e7_id_preds, e7_ood_preds = run_finetuned_experiment(
    experiment_id="E7",
    experiment_name="NLI Fine-Tuned",
    model_name=NLI_MODEL,
    train_data=train_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 19. E8: NLI + Keyword Masking + Counterfactual Augmentation

This experiment combines NLI initialization with keyword masking and counterfactual augmentation.

Model:

`cross-encoder/nli-roberta-base`

Training data:

`data/processed/train_full.csv`

In [ ]:
e8_metrics, e8_id_preds, e8_ood_preds = run_finetuned_experiment(
    experiment_id="E8",
    experiment_name="NLI + Keyword Masking + Counterfactual Augmentation",
    model_name=NLI_MODEL,
    train_data=train_full_df,
    use_replay=False,
    run_training=True,
    run_shap=True,
)

# 20. E9: NLI + Experience Replay + Keyword Masking + Counterfactual Augmentation

This is the strongest NLI-based experiment.

It combines:

- NLI initialization
- Keyword masking
- Counterfactual augmentation
- Experience replay

Model:

`cross-encoder/nli-roberta-base`

Training data:

`data/processed/train_full.csv`

Replay data:

`data/processed/replay_examples.csv`

In [ ]:
e9_metrics, e9_id_preds, e9_ood_preds = run_finetuned_experiment(
    experiment_id="E9",
    experiment_name="NLI + Experience Replay + Keyword Masking + Counterfactual Augmentation",
    model_name=NLI_MODEL,
    train_data=train_full_df,
    use_replay=True,
    replay_data=replay_df,
    replay_repeat=5,
    run_training=True,
    run_shap=True,
)

# 21. Final Model Comparison Table

In [ ]:
def load_experiment_summary(experiment_id):
    metrics = load_json(METRICS_DIR / f"{experiment_id}.json")

    if metrics is None:
        return {
            "experiment_id": experiment_id,
            "experiment_name": None,
            "id_accuracy": None,
            "id_macro_f1": None,
            "ood_accuracy": None,
            "ood_macro_f1": None,
            "id_ood_gap": None,
            "ood_confident_wrong_count": None,
        }

    id_acc = metrics.get("id_test", {}).get("accuracy")
    id_f1 = metrics.get("id_test", {}).get("macro_f1")
    ood_acc = metrics.get("ood", {}).get("accuracy")
    ood_f1 = metrics.get("ood", {}).get("macro_f1")

    gap = None

    if id_f1 is not None and ood_f1 is not None:
        gap = id_f1 - ood_f1

    return {
        "experiment_id": experiment_id,
        "experiment_name": metrics.get("experiment_name"),
        "id_accuracy": id_acc,
        "id_macro_f1": id_f1,
        "ood_accuracy": ood_acc,
        "ood_macro_f1": ood_f1,
        "id_ood_gap": gap,
        "ood_confident_wrong_count": metrics.get("ood", {}).get("confident_wrong_count"),
    }


experiment_ids = ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9"]

comparison_df = pd.DataFrame([
    load_experiment_summary(exp_id)
    for exp_id in experiment_ids
])

display(comparison_df)

comparison_path = RESULTS_DIR / "final_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(f"Saved comparison table to: {comparison_path}")

# 22. Final Comparison Visualizations

In [ ]:
plot_df = comparison_df.dropna(subset=["ood_macro_f1"]).copy()

plt.figure(figsize=(10, 4))
plt.bar(plot_df["experiment_id"], plot_df["ood_macro_f1"])
plt.title("OOD Macro-F1 by Experiment")
plt.xlabel("Experiment")
plt.ylabel("OOD Macro-F1")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ood_macro_f1_by_experiment.png", dpi=200, bbox_inches="tight")
plt.show()


gap_df = comparison_df.dropna(subset=["id_ood_gap"]).copy()

plt.figure(figsize=(10, 4))
plt.bar(gap_df["experiment_id"], gap_df["id_ood_gap"])
plt.title("ID-OOD Gap by Experiment")
plt.xlabel("Experiment")
plt.ylabel("ID Macro-F1 - OOD Macro-F1")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "id_ood_gap_by_experiment.png", dpi=200, bbox_inches="tight")
plt.show()


cw_df = comparison_df.dropna(subset=["ood_confident_wrong_count"]).copy()

plt.figure(figsize=(10, 4))
plt.bar(cw_df["experiment_id"], cw_df["ood_confident_wrong_count"])
plt.title("OOD Confident Wrong Count by Experiment")
plt.xlabel("Experiment")
plt.ylabel("Confident Wrong Count")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "ood_confident_wrong_count_by_experiment.png", dpi=200, bbox_inches="tight")
plt.show()

# 23. Best Model Summary

In [ ]:
print_experiment_header("Best Model Summary")

if comparison_df["ood_macro_f1"].notna().any():
    best_ood = comparison_df.sort_values("ood_macro_f1", ascending=False).head(1)
    print("Best model by OOD Macro-F1:")
    display(best_ood)

if comparison_df["id_ood_gap"].notna().any():
    best_gap = comparison_df.sort_values("id_ood_gap", ascending=True).head(1)
    print("Best model by lowest ID-OOD gap:")
    display(best_gap)

if comparison_df["ood_confident_wrong_count"].notna().any():
    best_cw = comparison_df.sort_values("ood_confident_wrong_count", ascending=True).head(1)
    print("Best model by lowest confident wrong count:")
    display(best_cw)

# 24. Display Saved Plots

In [ ]:
def display_all_pngs(folder):
    folder = Path(folder)

    if not folder.exists():
        print(f"Folder does not exist: {folder}")
        return

    png_files = sorted(folder.glob("*.png"))

    if not png_files:
        print(f"No PNG files found in {folder}")
        return

    for img_path in png_files:
        print(img_path)
        display(Image(filename=str(img_path)))


print("Final plots:")
display_all_pngs(PLOTS_DIR)